# 01 · Data Exploration — Civil Unrest Event Extraction & Prediction

**Project:** predicting whether protests/riots turn violent from text, using a transformer.
**This notebook:** understand what is actually in `data/` before designing labels, splits and models.

**What we want to learn**
1. What does each row represent, and what text do we have for the transformer?
2. How are *peaceful vs. violent* events distributed (class balance) — and how should we define the label?
3. How do events vary over time, space and actors?
4. What does the `notes` text look like (length, vocabulary, class-specific words, leakage risk)?
5. Is there a usable *escalation* signal at region × time-window level (peaceful activity now → violence next week)?

## 0. Setup

In [ ]:
import re, glob, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 200)
pd.set_option('display.width', 200)
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['figure.dpi'] = 110

# Works whether the notebook is launched from notebooks/ or from the project root
CANDIDATES = [Path('../data'), Path('data'), Path('NLP/data')]
DATA_DIR = next(p for p in CANDIDATES if p.exists())
print('Data dir:', DATA_DIR.resolve())
for f in sorted(DATA_DIR.glob('*.csv')):
    print(f'  {f.name}  ({f.stat().st_size/1e6:.1f} MB)')

## 1. Load

In [ ]:
csv_files = sorted(DATA_DIR.glob('*.csv'))
frames = []
for f in csv_files:
    d = pd.read_csv(f, low_memory=False, encoding='utf-8-sig')
    d['_source_file'] = f.name
    frames.append(d)
    print(f'{f.name[:60]:60s} -> {d.shape}')

df = pd.concat(frames, ignore_index=True)
df['event_date'] = pd.to_datetime(df['event_date'])
print('\nCombined:', df.shape)
print('Date range:', df.event_date.min().date(), '->', df.event_date.max().date())
df.head(3).T

## 2. Structure, types, missingness, duplicates

In [ ]:
info = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'n_unique': df.nunique(),
    'pct_missing': (df.isna().mean() * 100).round(1),
})
info

In [ ]:
miss = df.isna().mean().sort_values(ascending=False)
miss = miss[miss > 0]
ax = miss.mul(100).plot.barh(color='#4C72B0')
ax.set_xlabel('% missing'); ax.set_title('Columns with missing values'); ax.invert_yaxis()
plt.tight_layout(); plt.show()

print('Duplicate event ids :', df.event_id_cnty.duplicated().sum())
print('Duplicate notes text:', df.notes.duplicated().sum(), f'({df.notes.duplicated().mean():.2%})')
print('Countries           :', df.country.value_counts().to_dict())

## 3. Event taxonomy and label design

ACLED's hierarchy: `disorder_type` → `event_type` → `sub_event_type`. For *“will it turn violent?”* the relevant population is **Protests** and **Riots**.

In [ ]:
print(df.event_type.value_counts(), '\n')
ct = df.groupby(['event_type', 'sub_event_type']).size().rename('count').to_frame()
ct['pct_of_all'] = (ct['count'] / len(df) * 100).round(2)
ct

In [ ]:
# Fatalities by sub-event type
g = df.groupby('sub_event_type').agg(
    n=('event_id_cnty', 'size'),
    pct_with_fatalities=('fatalities', lambda s: round((s > 0).mean() * 100, 1)),
    total_fatalities=('fatalities', 'sum'),
).sort_values('n', ascending=False)
g

### Proposed label (to revisit)
Restrict to **Protests + Riots** (crowd events) and derive:

| `crowd_label` | meaning | ACLED sub-event types |
|---|---|---|
| 0 – peaceful | no violence, no intervention | Peaceful protest |
| 1 – tension | authorities intervene / use force | Protest with intervention, Excessive force against protesters |
| 2 – violent | crowd violence | Violent demonstration, Mob violence |

`is_violent` (binary) = `crowd_label == 2`. An alternative binary is `fatalities > 0`; both are compared below.

In [ ]:
crowd = df[df.event_type.isin(['Protests', 'Riots'])].copy()

def to_label(s):
    if s == 'Peaceful protest':
        return 0
    if s in ('Protest with intervention', 'Excessive force against protesters'):
        return 1
    return 2   # Violent demonstration, Mob violence

crowd['crowd_label'] = crowd.sub_event_type.map(to_label)
crowd['is_violent'] = (crowd.crowd_label == 2).astype(int)
crowd['has_fatality'] = (crowd.fatalities > 0).astype(int)

print('Crowd events:', len(crowd), f'({len(crowd)/len(df):.1%} of all rows)\n')
lab = crowd.crowd_label.map({0: '0 peaceful', 1: '1 tension', 2: '2 violent'}).value_counts().sort_index()
print(lab.to_string(), '\n')
print(f'Violent share (is_violent):  {crowd.is_violent.mean():.2%}')
print(f'Fatality share:              {crowd.has_fatality.mean():.2%}')
print('\nOverlap of the two binary definitions:')
print(pd.crosstab(crowd.is_violent, crowd.has_fatality, margins=True))

ax = lab.plot.bar(color=['#55A868', '#DD8452', '#C44E52'], rot=0)
ax.set_title('Crowd-event class balance'); ax.set_ylabel('events')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width()/2, p.get_height()), ha='center', va='bottom')
plt.tight_layout(); plt.show()

## 4. Time

In [ ]:
crowd['month'] = crowd.event_date.dt.to_period('M').dt.to_timestamp()
m = crowd.groupby(['month', 'crowd_label']).size().unstack(fill_value=0)
m.columns = ['peaceful', 'tension', 'violent']

fig, ax = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
m.plot.area(ax=ax[0], color=['#55A868', '#DD8452', '#C44E52'], alpha=.85)
ax[0].set_title('Crowd events per month'); ax[0].set_ylabel('events')
(m['violent'] / m.sum(axis=1) * 100).plot(ax=ax[1], color='#C44E52')
ax[1].set_title('Violent share per month (%)'); ax[1].set_ylabel('%'); ax[1].set_xlabel('')
plt.tight_layout(); plt.show()

In [ ]:
yearly = crowd.groupby('year').agg(n=('is_violent', 'size'), violent_rate=('is_violent', 'mean'),
                                   fatality_rate=('has_fatality', 'mean')).round(4)
display(yearly)

dow = crowd.groupby(crowd.event_date.dt.day_name()).is_violent.agg(['size', 'mean'])
dow = dow.reindex(['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'])
dow.columns = ['n', 'violent_rate']
display(dow.round(4))

# Busiest single days (nationwide protest waves)
daily = crowd.groupby('event_date').size()
print('\nTop 10 busiest days:'); print(daily.sort_values(ascending=False).head(10).to_string())

## 5. Geography

In [ ]:
st = crowd.groupby('admin1').agg(n=('is_violent', 'size'), violent_rate=('is_violent', 'mean')).sort_values('n', ascending=False)
print('States/UTs:', len(st))
fig, ax = plt.subplots(1, 2, figsize=(13, 6))
st.head(15)['n'].iloc[::-1].plot.barh(ax=ax[0], color='#4C72B0'); ax[0].set_title('Top 15 admin1 by crowd events')
st[st.n >= 500].sort_values('violent_rate').tail(15)['violent_rate'].mul(100).plot.barh(ax=ax[1], color='#C44E52')
ax[1].set_title('Highest violent share (admin1 with ≥ 500 events)'); ax[1].set_xlabel('% violent')
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
s = crowd.sample(min(60000, len(crowd)), random_state=0)
ax.scatter(s[s.is_violent == 0].longitude, s[s.is_violent == 0].latitude, s=2, alpha=.25, c='#55A868', label='non-violent')
ax.scatter(s[s.is_violent == 1].longitude, s[s.is_violent == 1].latitude, s=3, alpha=.5, c='#C44E52', label='violent')
ax.set_title('Event locations (sample)'); ax.set_xlabel('lon'); ax.set_ylabel('lat'); ax.legend(markerscale=5)
plt.tight_layout(); plt.show()

print('Unique admin1 / admin2 / location:', crowd.admin1.nunique(), crowd.admin2.nunique(), crowd.location.nunique())

## 6. Actors, scale and metadata

In [ ]:
for c in ['inter1', 'interaction', 'source_scale', 'time_precision', 'geo_precision']:
    t = crowd.groupby(c).is_violent.agg(['size', 'mean']).sort_values('size', ascending=False).head(10)
    t.columns = ['n', 'violent_rate']
    print(f'--- {c}'); display(t.round(4))

In [ ]:
# `tags` is a ';'-separated bag of flags, including 'crowd size=...' (a scale signal for the 'scale' argument)
def parse_tags(s):
    return [t.strip() for t in str(s).split(';')] if pd.notna(s) else []

crowd['tag_list'] = crowd.tags.map(parse_tags)
crowd['crowd_size'] = crowd.tag_list.map(lambda L: next((t.split('=')[1] for t in L if t.startswith('crowd size=')), np.nan))

print('Tag frequency:'); print(Counter(t for L in crowd.tag_list for t in L if not t.startswith('crowd size=')).most_common(15))
print(f'\nRows with a crowd-size tag: {crowd.crowd_size.notna().mean():.1%}')
order = ['very small', 'small', 'medium', 'large', 'massive']
cs = crowd.groupby('crowd_size').is_violent.agg(['size', 'mean']).reindex(order); cs.columns = ['n', 'violent_rate']
display(cs.round(4))

In [ ]:
# Top actors (actor1) and their violent rate
a = crowd.groupby('actor1').is_violent.agg(['size', 'mean']).sort_values('size', ascending=False).head(15)
a.columns = ['n', 'violent_rate']; a.round(4)

## 7. Fatalities

In [ ]:
print(crowd.fatalities.describe().round(3).to_string())
print('\nEvents with ≥1 fatality by class:')
print(crowd.groupby('crowd_label').has_fatality.agg(['size', 'sum', 'mean']).round(4))
print('\nFatality count distribution (events with fatalities):')
print(crowd[crowd.has_fatality == 1].fatalities.value_counts().sort_index().head(10).to_string())

## 8. Text — the `notes` field

This is the only free text and what a transformer will read. **Important:** ACLED `notes` are *analyst-written summaries of news reports*, not raw social-media posts or conversations — see the limitations section at the end.

In [ ]:
crowd['notes'] = crowd['notes'].fillna('')
crowd['n_chars'] = crowd.notes.str.len()
crowd['n_words'] = crowd.notes.str.split().str.len()

print(crowd[['n_chars', 'n_words']].describe(percentiles=[.5, .9, .95, .99]).round(1))
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
sns.histplot(crowd.n_words.clip(upper=150), bins=50, ax=ax[0], color='#4C72B0'); ax[0].set_title('Words per note (clipped at 150)')
sns.boxplot(data=crowd, x='crowd_label', y='n_words', showfliers=False, ax=ax[1], palette=['#55A868', '#DD8452', '#C44E52'])
ax[1].set_title('Note length by class'); plt.tight_layout(); plt.show()
print('→ 95% of notes have <=', int(crowd.n_words.quantile(.95)), 'words; max_length=128 tokens is probably enough (verify with the real tokenizer).')

In [ ]:
# Read a few examples per class
for lbl, name in [(0, 'PEACEFUL'), (1, 'TENSION'), (2, 'VIOLENT')]:
    print(f'===== {name} =====')
    for t in crowd[crowd.crowd_label == lbl].notes.sample(3, random_state=1):
        print('•', t, '\n')

In [ ]:
STOP = set('''a an the and or of in on at to for with by from as is was were be been are it its this that these those
their his her they he she them who which while after before over under into during also but not no than then there
has had have said says per near about against among between around up out us we our more most other another one two
three four some any all being upon due'''.split())

def tok(s):
    return [w for w in re.findall(r"[a-z']+", s.lower()) if w not in STOP and len(w) > 2]

cnt = {k: Counter() for k in (0, 1, 2)}
for lbl, txt in zip(crowd.crowd_label, crowd.notes):
    cnt[lbl].update(set(tok(txt)))   # document frequency, not raw count
n_docs = crowd.crowd_label.value_counts().to_dict()

print('Top 20 words per class (document frequency %):')
top = pd.DataFrame({name: [f'{w} ({c/n_docs[k]:.0%})' for w, c in cnt[k].most_common(20)]
                    for k, name in [(0, 'peaceful'), (1, 'tension'), (2, 'violent')]})
top

In [ ]:
# Words that best separate violent vs. peaceful (smoothed log-odds ratio)
def log_odds(c_a, c_b, n_a, n_b, min_df=150, alpha=1.0):
    rows = []
    for w in set(c_a) | set(c_b):
        a, b = c_a.get(w, 0), c_b.get(w, 0)
        if a + b < min_df: continue
        lo = np.log((a + alpha) / (n_a - a + alpha)) - np.log((b + alpha) / (n_b - b + alpha))
        rows.append((w, a, b, lo))
    return pd.DataFrame(rows, columns=['word', 'df_violent', 'df_peaceful', 'log_odds']).sort_values('log_odds')

lo = log_odds(cnt[2], cnt[0], n_docs[2], n_docs[0])
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
lo.tail(20).set_index('word').log_odds.plot.barh(ax=ax[0], color='#C44E52'); ax[0].set_title('Most violent-indicative words')
lo.head(20).set_index('word').log_odds.plot.barh(ax=ax[1], color='#55A868'); ax[1].set_title('Most peaceful-indicative words')
plt.tight_layout(); plt.show()

### Trigger words and leakage check
Our proposal lists trigger words such as *protest, march, clash, strike*. We check (a) how common they are and (b) whether the **label is almost spelled out in the text** — if so, a model can score highly by keyword matching and not by understanding escalation (label leakage).

In [ ]:
TRIGGERS = ['protest', 'march', 'rally', 'demonstrat', 'strike', 'blockade', 'dharna', 'bandh', 'clash', 'stone', 'pelt',
            'violence', 'violent', 'riot', 'arson', 'torch', 'vandal', 'set fire', 'lynch', 'injur', 'killed', 'lathi', 'tear gas',
            'detain', 'arrest', 'police']
low = crowd.notes.str.lower()
rows = []
for t in TRIGGERS:
    hit = low.str.contains(t, regex=False)
    rows.append((t, hit.mean(), crowd.loc[hit, 'is_violent'].mean() if hit.any() else np.nan,
                 crowd.loc[~hit, 'is_violent'].mean()))
trig = pd.DataFrame(rows, columns=['trigger', 'coverage', 'violent_rate_if_present', 'violent_rate_if_absent']).set_index('trigger')
trig.round(3).sort_values('violent_rate_if_present', ascending=False)

In [ ]:
# How well does a trivial keyword rule recover the label? (a cheap baseline the transformer must beat)
kw = ['clash', 'stone', 'pelt', 'violen', 'riot', 'arson', 'torch', 'vandal', 'set fire', 'lynch', 'injur', 'killed', 'ransack', 'attack', 'beat', 'thrash']
pred = low.apply(lambda s: any(k in s for k in kw)).astype(int)
tp = ((pred == 1) & (crowd.is_violent == 1)).sum(); fp = ((pred == 1) & (crowd.is_violent == 0)).sum()
fn = ((pred == 0) & (crowd.is_violent == 1)).sum()
prec, rec = tp / (tp + fp), tp / (tp + fn)
print(f'Keyword baseline  precision={prec:.3f}  recall={rec:.3f}  F1={2*prec*rec/(prec+rec):.3f}')

# Duplicate text (matters for train/test leakage)
dup = crowd.notes.duplicated(keep=False)
print(f'\nRows sharing identical notes with another row: {dup.sum()} ({dup.mean():.2%})')
print(crowd.notes.str.slice(0, 80).value_counts().head(5).to_string())

In [ ]:
# Character-level sanity: non-ASCII, URLs, empty notes
print('Empty notes   :', (crowd.notes.str.len() == 0).sum())
print('Non-ASCII rows:', crowd.notes.map(lambda s: any(ord(c) > 127 for c in s)).mean().round(4))
print('Contains URL  :', crowd.notes.str.contains('http').mean().round(4))

## 9. Escalation signal — region × week panel

The second research goal is *forecasting*: can past activity in a region predict **violence in the following window**? We aggregate to `admin1 × week` and measure how strongly current protest/riot activity relates to next-week violence. This is a preliminary look, not a model.

In [ ]:
crowd['week'] = crowd.event_date.dt.to_period('W-SUN').dt.start_time
panel = (crowd.groupby(['admin1', 'week'])
              .agg(n_events=('is_violent', 'size'), n_violent=('is_violent', 'sum'),
                   n_tension=('crowd_label', lambda s: (s == 1).sum()),
                   n_fatal=('has_fatality', 'sum'))
              .reset_index())

# Fill missing weeks with zeros so "next week" is truly the next calendar week
full_idx = pd.MultiIndex.from_product([panel.admin1.unique(), pd.date_range(panel.week.min(), panel.week.max(), freq='W-MON')],
                                      names=['admin1', 'week'])
panel = panel.set_index(['admin1', 'week']).reindex(full_idx, fill_value=0).reset_index()
panel = panel.sort_values(['admin1', 'week'])
panel['n_peaceful'] = panel.n_events - panel.n_violent - panel.n_tension
panel['violent_next'] = (panel.groupby('admin1').n_violent.shift(-1) > 0).astype(float)
panel['violent_now'] = (panel.n_violent > 0).astype(int)
panel = panel.dropna(subset=['violent_next'])
panel['violent_next'] = panel.violent_next.astype(int)

print('Panel:', panel.shape, f'| admin1 units: {panel.admin1.nunique()} | weeks: {panel.week.nunique()}')
print(f'Base rate P(violent event next week): {panel.violent_next.mean():.2%}')
print(f'P(next | violent this week)    : {panel[panel.violent_now == 1].violent_next.mean():.2%}')
print(f'P(next | no violence this week): {panel[panel.violent_now == 0].violent_next.mean():.2%}')

In [ ]:
bins = pd.cut(panel.n_peaceful, [-1, 0, 2, 5, 10, 25, 1e9], labels=['0', '1-2', '3-5', '6-10', '11-25', '25+'])
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
panel.groupby(bins).violent_next.mean().mul(100).plot.bar(ax=ax[0], color='#4C72B0', rot=0)
ax[0].set_title('P(violence next week) by # peaceful protests this week'); ax[0].set_ylabel('%'); ax[0].set_xlabel('peaceful protests this week')
panel.groupby('violent_now').violent_next.mean().mul(100).rename({0: 'no violence', 1: 'violence'}).plot.bar(ax=ax[1], color='#C44E52', rot=0)
ax[1].set_title('P(violence next week) by violence this week'); ax[1].set_ylabel('%'); ax[1].set_xlabel('')
plt.tight_layout(); plt.show()

In [ ]:
# Lag structure: autocorrelation of weekly violent-event counts for the 6 busiest units
top_units = panel.groupby('admin1').n_events.sum().sort_values(ascending=False).head(6).index
ac = pd.DataFrame({u: [panel[panel.admin1 == u].n_violent.autocorr(l) for l in range(1, 9)] for u in top_units}, index=range(1, 9))
ac.index.name = 'lag (weeks)'
ax = ac.plot(marker='o', figsize=(9, 4)); ax.set_title('Autocorrelation of weekly violent events'); ax.set_ylabel('r')
plt.tight_layout(); plt.show()

## 10. Save the derived crowd-event table for later notebooks

In [ ]:
OUT = DATA_DIR / 'processed'
OUT.mkdir(exist_ok=True)
keep = ['event_id_cnty', 'event_date', 'admin1', 'admin2', 'location', 'latitude', 'longitude', 'event_type', 'sub_event_type',
        'actor1', 'actor2', 'interaction', 'fatalities', 'crowd_size', 'notes', 'crowd_label', 'is_violent', 'has_fatality']
crowd[keep].to_csv(OUT / 'crowd_events.csv', index=False)
print('Saved', (OUT / 'crowd_events.csv').resolve(), crowd[keep].shape)

## 11. Findings & limitations (read before modelling)

**What the data is.** A single ACLED export covering **India only, 2018-01-01 → 2025-11-01**, ~168 k events (≈ 149 k Protests + Riots). Each row is one *event* with a short analyst-written `notes` summary, structured actors, location, `crowd size` tags and fatality counts.

**Key observations**
- **Heavy class imbalance:** most crowd events are *peaceful protests*; violent events (Riots) are a minority → use time-aware stratified splits, class weights / focal loss and report PR-AUC / macro-F1, not accuracy.
- **Rich structure for the extraction half of the project:** actor1/actor2 (organizer, target), admin1/admin2/location, date and crowd-size tag map to the planned arguments (actor, location, time, scale). Trigger words are present in `notes`.
- **Escalation signal at region-week level:** see Section 9 — violence is autocorrelated and conditional rates differ by activity level, so a sequence model over region-week aggregates is reasonable.

**Limitations / risks**
1. **Not social-media conversation.** `notes` are news-derived summaries in a neutral reporting style. The project targets informal Twitter/Facebook/Reddit text; ACLED alone cannot teach a model slang, sarcasm or emotion. A social-media corpus must be added, with ACLED used for **weak labels / ground truth** (as the proposal says).
2. **No sentiment labels.** There is no sentiment/emotion column. Options: weak-label with an off-the-shelf sentiment/emotion model, hand-annotate a small sample, or use sentiment as an auxiliary feature rather than a target.
3. **Label leakage risk.** `sub_event_type` (our label) was *derived by ACLED coders from the same text*, so the notes carry lexical cues (“clashed”, “stone pelting”, “injured”). A hand-written keyword rule only reaches F1 ≈ 0.65 (Section 8), but a plain TF-IDF + logistic regression (`scripts/train_baseline.py`) reaches violent-class F1 ≈ 0.97 on the time-based test split, and still ≈ 0.96 with the obvious cue words masked — the coders' decision is recoverable from wording such as “rioters”, “mob”, “burnt tyres”. So classifying an event from **its own description** is close to solved, and a fine-tuned transformer will mostly match that ceiling. The harder, more interesting question is **forecasting** escalation from *earlier* events/text (Section 9).
4. **Duplicate/templated notes** exist; split by time (and/or by region), never randomly.
5. **Single country, English only, news-source bias** (`source_scale`) — results will not generalise to other regions or languages without more data.
6. **Reporting artefacts:** counts vary with media coverage and coding-policy changes (see year-level shifts in Section 4).

**Suggested next notebooks:** `02_label_definition_and_splits` → `03_keyword_and_tfidf_baselines` → `04_transformer_finetuning` → `05_event_sequence_forecasting`.